# Stepageddon - Train Step Chart Model

Train a neural network to generate DDR step charts from audio.

**Prerequisites:**
1. Run `prepare_data.py` locally to preprocess your charts
2. Zip the training data: `cd backend/ml && zip -r training_data.zip training_data/`
3. Upload `training_data.zip` directly to Colab when prompted (cell below)

**Runtime:** Select GPU (T4) under Runtime > Change runtime type

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q librosa numpy torch

In [ ]:
# Upload the ml/ module code
from google.colab import files
import os

os.makedirs('/content/ml', exist_ok=True)
# Make /content/ml a real Python package so `import ml.train` works
open('/content/ml/__init__.py', 'w').close()

print('Upload ALL of the following from backend/ml/:')
print('  model.py, dataset.py, train.py, prepare_data.py, sm_parser.py, inference.py')
uploaded = files.upload()
for name, data in uploaded.items():
    out_path = os.path.join('/content/ml', os.path.basename(name))
    with open(out_path, 'wb') as f:
        f.write(data)
    print(f'Wrote {out_path}')

!ls -la /content/ml/


In [ ]:
# Upload and extract training data
# Pick ONE option below and comment out the others

# --- Option A: Copy from Google Drive to local disk (most reliable for large files) ---
# Upload training_data.zip to your Google Drive first (any location), then:
import zipfile, os
ZIP_PATH = '/content/drive/MyDrive/stepageddon/training_data.zip'  # adjust path if needed
print(f'Copying and extracting from Drive...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
print('Done!')

# --- Option B: Direct URL download (if hosted somewhere) ---
# !wget -q -O /content/training_data.zip "YOUR_URL_HERE"
# import zipfile
# with zipfile.ZipFile('/content/training_data.zip', 'r') as z:
#     z.extractall('/content/')
# !rm /content/training_data.zip

# --- Option C: Browser upload (only works for small files <1GB) ---
# from google.colab import files
# uploaded = files.upload()
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
#     z.extractall('/content/')

!ls /content/training_data/ | head -20

In [ ]:
# Verify data is accessible
import json
from pathlib import Path

DATA_DIR = Path('/content/training_data')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/stepageddon/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

print(f'Total training examples: {len(manifest)}')

# Show difficulty distribution
from collections import Counter
diff_counts = Counter(e['difficulty'] for e in manifest)
print(f'Difficulty distribution: {dict(diff_counts)}')

# Check a sample file
import numpy as np
sample = np.load(DATA_DIR / manifest[0]['filename'])
print(f"\nSample: {manifest[0]['song_title']} ({manifest[0]['difficulty']})")
print(f'  Mel shape: {sample["mel"].shape}')
print(f'  Labels shape: {sample["labels"].shape}')
print(f'  Note frames: {(sample["labels"] > 0).any(axis=1).sum()}')

In [ ]:
# Add to Python path
import sys
sys.path.insert(0, '/content')

# Also need the modules for schema imports during inference (not needed for training)
# For training only, we just need ml.model and ml.dataset

In [ ]:
# Train the model by invoking ml.train as a module. This keeps the
# notebook in sync with the canonical CLI training script — any future
# changes to the training loop (EMA, curriculum, dynamic class weights,
# tolerant F1 metrics, etc.) are picked up automatically.
#
# Adjust hyperparameters via command-line flags instead of editing the
# notebook. See `python -m ml.train --help` for the full list.

import sys
sys.path.insert(0, '/content')

DATA_DIR = '/content/training_data'
CHECKPOINT_DIR = '/content/drive/MyDrive/stepageddon/checkpoints'

!python -m ml.train \
    --data-dir {DATA_DIR} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs 80 \
    --batch-size 32 \
    --lr 3e-4 \
    --hidden-dim 256 \
    --n-heads 8 \
    --n-layers 4 \
    --chunk-frames 500 \
    --samples-per-entry 2 \
    --val-split 0.1 \
    --warmup-epochs 5 \
    --curriculum-epochs 10 \
    --curriculum-easy-ids 0,1,2 \
    --ema-decay 0.999 \
    --early-stop-patience 8


## After Training

1. Download `best_model.pt` from Google Drive (`stepageddon/checkpoints/best_model.pt`)
2. Place it in `backend/ml/checkpoints/best_model.pt`
3. Set `USE_ML_GENERATION=true` in `backend/.env`
4. Restart the backend server